In [14]:
"""
Interfaz grafica (Tkinter) para graphene_wakefield_nlayer.py
==============================================================
Permite configurar:
  - Las capas de grafeno (anadir / quitar filas), cada una con su
    posicion z (nm), densidad n0/n_g y amortiguamiento gamma (u.a.)
  - El driver (carga puntual, siempre sobre el eje y=0): v/c, Q, z0
  - El sustrato dielectrico opcional: eps_s, z_s
  - La malla en espacio-k (kx_max, ky_max, n_kx, n_ky) que controla
    la resolucion/convergencia del calculo
  - El rango de salida en espacio real (zeta, z, y) para las graficas

y un boton "Ejecutar" que llama a MultilayerGraphene y dibuja los
mapas de color de los campos de estela (Wx, Wz en el plano zeta-z;
Wx, Wy, Wz en el plano zeta-y), embebidos en la ventana.

Requisitos: numpy, matplotlib. El archivo
graphene_wakefield_nlayer.py debe estar en la misma carpeta.

Ejecutar con:  python3 interfaz_wakefields.py
"""

import sys
import threading
import traceback
import tkinter as tk
from tkinter import ttk, messagebox, filedialog


def _make_dpi_aware():
    if sys.platform.startswith("win"):
        import ctypes
        try:
            ctypes.windll.shcore.SetProcessDpiAwareness(1)  # PROCESS_SYSTEM_DPI_AWARE
        except Exception:
            try:
                ctypes.windll.user32.SetProcessDPIAware()
            except Exception:
                pass

import numpy as np
import matplotlib
matplotlib.use("TkAgg")
import matplotlib.pyplot as plt
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg, NavigationToolbar2Tk
from matplotlib.colors import CenteredNorm

from graphene_wakefield_nlayer import MultilayerGraphene
from graphene_simple import perturbed_density_zeta_y_adaptive


# --------------------------------------------------------------------- utils
def parse_float(entry, name, default=None):
    txt = entry.get().strip()
    if txt == "" and default is not None:
        return default
    try:
        return float(txt)
    except ValueError:
        raise ValueError(f"'{name}' no es un numero valido: '{txt}'")


def parse_int(entry, name, default=None):
    txt = entry.get().strip()
    if txt == "" and default is not None:
        return default
    try:
        return int(float(txt))
    except ValueError:
        raise ValueError(f"'{name}' no es un entero valido: '{txt}'")


# --------------------------------------------------------------- fila capa
class LayerRow:
    """Una fila editable de la tabla de capas: z (nm), n0/n_g, gamma (u.a.)."""

    def __init__(self, parent, on_remove, z=-1.0, n0=1.0, gamma=1e-3):
        self.frame = ttk.Frame(parent)
        self.z_var = tk.StringVar(value=str(z))
        self.n0_var = tk.StringVar(value=str(n0))
        self.gamma_var = tk.StringVar(value=str(gamma))

        ttk.Entry(self.frame, textvariable=self.z_var, width=10).grid(row=0, column=0, padx=2)
        ttk.Entry(self.frame, textvariable=self.n0_var, width=10).grid(row=0, column=1, padx=2)
        ttk.Entry(self.frame, textvariable=self.gamma_var, width=10).grid(row=0, column=2, padx=2)
        ttk.Button(self.frame, text="Quitar", width=8,
                   command=lambda: on_remove(self)).grid(row=0, column=3, padx=4)

        self.frame.pack(fill="x", pady=1)

    def destroy(self):
        self.frame.destroy()

    def values(self):
        try:
            z = float(self.z_var.get())
            n0 = float(self.n0_var.get())
            gamma = float(self.gamma_var.get())
        except ValueError:
            raise ValueError("Todas las capas necesitan valores numericos "
                              "(z, n0/n_g, gamma).")
        return z, n0, gamma


# --------------------------------------------------------------- scrollbar
class CustomScrollbar(tk.Frame):

    def __init__(self, parent, target_canvas, width=32,
                 trough_color="#c4c4c4", thumb_color="#6b6b6b",
                 thumb_active_color="#4a4a4a", min_thumb_height=24):
        super().__init__(parent, width=width)
        self.target_canvas = target_canvas
        self.width = width
        self.min_thumb_height = min_thumb_height
        self._lo, self._hi = 0.0, 1.0
        self._drag_offset = None

        self.track = tk.Canvas(self, width=width, bg=trough_color,
                                highlightthickness=0, bd=0)
        self.track.pack(fill="both", expand=True)
        self.thumb_color = thumb_color
        self.thumb = self.track.create_rectangle(
            2, 0, width - 2, min_thumb_height, fill=thumb_color, outline="")

        self.track.bind("<Configure>", lambda e: self._redraw())
        self.track.bind("<Button-1>", self._on_track_press)
        self.track.bind("<B1-Motion>", self._on_drag)
        self.track.bind("<ButtonRelease-1>", self._on_release)
        self.track.tag_bind(self.thumb, "<Enter>",
                             lambda e: self.track.itemconfig(self.thumb, fill=thumb_active_color))
        self.track.tag_bind(self.thumb, "<Leave>",
                             lambda e: self.track.itemconfig(self.thumb, fill=thumb_color))

        # rueda del raton tambien mueve el contenido
        self.track.bind("<MouseWheel>", self._on_mousewheel)

    # API compatible con el Scrollbar clasico: Canvas la llama via
    # yscrollcommand cada vez que cambia la vista visible.
    def set(self, lo, hi):
        self._lo, self._hi = float(lo), float(hi)
        self._redraw()

    def _redraw(self):
        h = self.track.winfo_height()
        if h <= 1:
            return
        top = self._lo * h
        bottom = self._hi * h
        if bottom - top < self.min_thumb_height:
            center = (top + bottom) / 2
            top = center - self.min_thumb_height / 2
            bottom = center + self.min_thumb_height / 2
        top = max(0, top)
        bottom = min(h, bottom)
        self.track.coords(self.thumb, 2, top, self.width - 2, bottom)

    def _on_track_press(self, event):
        coords = self.track.coords(self.thumb)
        if coords and coords[1] <= event.y <= coords[3]:
            self._drag_offset = event.y - coords[1]
        else:
            self._drag_offset = (coords[3] - coords[1]) / 2 if coords else 0
            self._move_thumb_to(event.y)

    def _on_drag(self, event):
        if self._drag_offset is None:
            return
        self._move_thumb_to(event.y)

    def _on_release(self, event):
        self._drag_offset = None

    def _move_thumb_to(self, y):
        h = self.track.winfo_height()
        coords = self.track.coords(self.thumb)
        thumb_h = (coords[3] - coords[1]) if coords else self.min_thumb_height
        top = y - (self._drag_offset if self._drag_offset is not None else thumb_h / 2)
        top = max(0, min(top, h - thumb_h))
        frac = top / (h - thumb_h) if h > thumb_h else 0.0
        self.target_canvas.yview_moveto(frac)

    def _on_mousewheel(self, event):
        self.target_canvas.yview_scroll(int(-1 * (event.delta / 120)), "units")


# ------------------------------------------------------------------- app
class WakefieldApp(tk.Tk):

    # Ancho (en pixeles) de las barras de scroll de cada panel.
    SCROLLBAR_WIDTH_LEFT = 32
    SCROLLBAR_WIDTH_RIGHT = 22

    def __init__(self):
        super().__init__()
        self.title("Wakefields en grafeno multicapa")
        self.geometry("1300x850")

        self.layer_rows = []
        self.last_figure = None

        self._build_layout()
        self._add_layer_row(-1.0, 1.0, 1e-3)
        self._add_layer_row(-2.0, 1.0, 1e-3)
        self._add_layer_row(-3.0, 1.0, 1e-3)

    # ---------------------------------------------------------- layout
    def _build_layout(self):
        outer = ttk.Frame(self)
        outer.pack(fill="both", expand=True)

        # panel izquierdo (parametros) con scroll arriba + boton Ejecutar
        # SIEMPRE visible fijo abajo (para que no quede oculto por muchas opciones)
        left_container = ttk.Frame(outer, width=430)
        left_container.pack(side="left", fill="y")
        left_container.pack_propagate(False)

        # zona fija abajo: se empaqueta primero con side="bottom" para que
        # siempre reserve su espacio, sin importar cuanto crezca el scroll
        f_run = ttk.Frame(left_container)
        f_run.pack(side="bottom", fill="x", padx=8, pady=6)
        ttk.Separator(left_container, orient="horizontal").pack(side="bottom", fill="x")
        self.btn_run = ttk.Button(f_run, text="Ejecutar", command=self._on_run)
        self.btn_run.pack(fill="x", pady=4)
        self.btn_save = ttk.Button(f_run, text="Guardar figura como PNG...",
                                    command=self._on_save, state="disabled")
        self.btn_save.pack(fill="x")
        self.progress = ttk.Progressbar(f_run, mode="indeterminate")
        self.progress.pack(fill="x", pady=4)
        self.status_var = tk.StringVar(value="Listo.")
        ttk.Label(f_run, textvariable=self.status_var, foreground="#444",
                  wraplength=340, justify="left").pack(fill="x")

        canvas = tk.Canvas(left_container, borderwidth=0, highlightthickness=0)
        # Barra de scroll dibujada a mano (ver clase CustomScrollbar):
        # el Scrollbar nativo de Windows ignora el ancho pedido.
        # Se coloca en el borde izquierdo del todo y se empaqueta ANTES
        # que el canvas, para que su ancho quede reservado de verdad
        # (si se empaqueta despues de un widget con expand=True, este
        # ultimo puede comerse todo el hueco y dejar la barra aplastada).
        vscroll = CustomScrollbar(left_container, canvas, width=self.SCROLLBAR_WIDTH_LEFT)
        vscroll.pack(side="left", fill="y")
        self.left = ttk.Frame(canvas)
        self.left.bind("<Configure>", lambda e: canvas.configure(scrollregion=canvas.bbox("all")))
        canvas.create_window((0, 0), window=self.left, anchor="nw")
        canvas.configure(yscrollcommand=vscroll.set)
        canvas.pack(side="left", fill="both", expand=True)
        canvas.bind("<MouseWheel>", vscroll._on_mousewheel)

        # panel derecho (graficas), tambien con scroll por si la figura
        # combinada (zz + zy + densidad en varias capas) queda muy alta
        right_container = ttk.Frame(outer)
        right_container.pack(side="left", fill="both", expand=True)
        right_canvas = tk.Canvas(right_container, borderwidth=0, highlightthickness=0)
        right_vscroll = tk.Scrollbar(right_container, orient="vertical",
                                      width=self.SCROLLBAR_WIDTH_RIGHT, command=right_canvas.yview)
        self.right = ttk.Frame(right_canvas)
        self.right.bind("<Configure>",
                         lambda e: right_canvas.configure(scrollregion=right_canvas.bbox("all")))
        right_canvas.create_window((0, 0), window=self.right, anchor="nw")
        right_canvas.configure(yscrollcommand=right_vscroll.set)
        right_canvas.pack(side="left", fill="both", expand=True)
        right_vscroll.pack(side="right", fill="y")

        pad = dict(padx=8, pady=4)

        # ---- Explicacion sencilla para el usuario ----
        f_intro = ttk.LabelFrame(self.left, text="MLGRAFMAKER")
        f_intro.pack(fill="x", **pad)
        ttk.Label(
            f_intro,
            text=(
                "Simula el campo electrico ( wakefield) que deja una partícula cargada\n"
                " al pasar cerca de una o varias capas de grafeno apiladas.\n"
                "1) Define las capas de grafeno y la carga que las atraviesa.\n"
                "2) Pulsa 'Ejecutar' para calcular los campos.\n"
            ),
            foreground="#333", justify="left", wraplength=340,
        ).pack(anchor="w", padx=4, pady=4)

        # ---- Capas de grafeno ----
        f_layers = ttk.LabelFrame(self.left, text="Capas de grafeno")
        f_layers.pack(fill="x", **pad)
        hdr = ttk.Frame(f_layers)
        hdr.pack(fill="x")
        for txt, w in (("z (nm)", 10), ("n0 / n_g", 10), ("gamma (u.a.)", 10)):
            ttk.Label(hdr, text=txt, width=w).pack(side="left", padx=2)
        self.layers_frame = ttk.Frame(f_layers)
        self.layers_frame.pack(fill="x")
        ttk.Button(f_layers, text="+ Anadir capa",
                   command=lambda: self._add_layer_row(-float(len(self.layer_rows) + 1), 1.0, 1e-3)
                   ).pack(pady=4)

        # ---- Driver ----
        f_drv = ttk.LabelFrame(self.left, text="Carga excitadora (driver)")
        f_drv.pack(fill="x", **pad)
        self.e_v = self._labeled_entry(f_drv, "v / c", "0.05")
        self.e_Q = self._labeled_entry(f_drv, "Q (carga, u.a.)", "1.0")
        self.e_z0 = self._labeled_entry(f_drv, "z0 (nm, altura driver)", "0.0")
        ttk.Label(f_drv, text="La particula viaja sobre el eje (y=0),\n"
                              ,
                  foreground="#666", justify="left").pack(anchor="w", padx=4, pady=(2, 4))

        # ---- Sustrato ----
        f_sub = ttk.LabelFrame(self.left, text="Sustrato dielectrico (opcional)")
        f_sub.pack(fill="x", **pad)
        self.var_sub = tk.BooleanVar(value=True)
        ttk.Checkbutton(f_sub, text="Incluir sustrato", variable=self.var_sub,
                         command=self._toggle_substrate).pack(anchor="w")
        self.e_eps = self._labeled_entry(f_sub, "eps_s (permitividad)", "3.9")
        self.e_zs = self._labeled_entry(f_sub, "z_s (nm, posicion, z<=zs)", "-3.5")

        # ---- Malla en espacio-k ----
        f_mesh = ttk.LabelFrame(self.left, text="Definición de la malla en espacio-k")
        f_mesh.pack(fill="x", **pad)
        self.e_kmax = self._labeled_entry(f_mesh, "k_max (u.a.)", "4.0")
        self.e_nkx = self._labeled_entry(f_mesh, "n_kx (puntos)", "600")
        self.e_nky = self._labeled_entry(f_mesh, "n_ky (puntos)", "300")
        ttk.Label(f_mesh, text=" Recomendable kx/ ky de al menos 2000 para evitar artefactos\n"
                                ,
                  foreground="#666", justify="left").pack(anchor="w", padx=4, pady=(2, 4))

        # ---- Que graficas calcular ----
        f_which = ttk.LabelFrame(self.left, text="Graficas a calcular")
        f_which.pack(fill="x", **pad)
        self.var_zz = tk.BooleanVar(value=True)
        self.var_zy = tk.BooleanVar(value=True)
        ttk.Checkbutton(f_which, text="Plano zeta-z (Wx, Wy, Wz), y=0", variable=self.var_zz).pack(anchor="w")
        ttk.Checkbutton(f_which, text="Plano zeta-y (Wx, Wy, Wz), z fijo", variable=self.var_zy).pack(anchor="w")
      

        # ---- Densidad de carga perturbada n1/n0 (cuadratura adaptativa) ----
        f_dens = ttk.LabelFrame(self.left, text="Densidad perturbada n1/n0 (adaptativa)")
        f_dens.pack(fill="x", **pad)
        self.var_dens = tk.BooleanVar(value=False)
        ttk.Checkbutton(f_dens, text="Calcular n1/n0", variable=self.var_dens,
                         command=self._toggle_dens).pack(anchor="w")

        self.var_dens_all = tk.BooleanVar(value=False)
        row_layer = ttk.Frame(f_dens)
        row_layer.pack(fill="x", padx=4, pady=2)
        self.chk_dens_all = ttk.Checkbutton(row_layer, text="Todas las capas",
                                             variable=self.var_dens_all,
                                             command=self._toggle_dens)
        self.chk_dens_all.pack(side="left")
        ttk.Label(row_layer, text="   capa indice:").pack(side="left")
        self.sp_dens_layer = ttk.Spinbox(row_layer, from_=0, to=99, width=5)
        self.sp_dens_layer.set(0)
        self.sp_dens_layer.pack(side="left")

        self.e_kmax_dens = self._labeled_entry(f_dens, "k_max (u.a.)", "2.5")
        self.e_nkx_dens = self._labeled_entry(f_dens, "n_kx (malla externa)", "150")
        self.e_epsabs = self._labeled_entry(f_dens, "epsabs (tol. absoluta)", "1e-8")
        self.e_epsrel = self._labeled_entry(f_dens, "epsrel (tol. relativa)", "1e-6")
        ttk.Label(f_dens,
                  text="Usa el mismo rango zeta/y (automatico) que el plano zeta-y.\n"
                       " ky se calcula adaptativamente para garantizar la convergencia\n"
                       " (o un error estimado).\n",
                  foreground="#666", justify="left").pack(anchor="w", padx=4, pady=(2, 4))

        self._toggle_substrate()

    def _labeled_entry(self, parent, label, default):
        row = ttk.Frame(parent)
        row.pack(fill="x", padx=4, pady=2)
        ttk.Label(row, text=label, width=20).pack(side="left")
        e = ttk.Entry(row, width=12)
        e.insert(0, default)
        e.pack(side="left", fill="x", expand=True)
        return e

    def _add_layer_row(self, z, n0, gamma):
        row = LayerRow(self.layers_frame, self._remove_layer_row, z, n0, gamma)
        self.layer_rows.append(row)

    def _remove_layer_row(self, row):
        if len(self.layer_rows) <= 1:
            messagebox.showwarning("Aviso", "Debe quedar al menos una capa.")
            return
        row.destroy()
        self.layer_rows.remove(row)

    def _toggle_dens(self):
        dens_on = self.var_dens.get()
        all_on = self.var_dens_all.get()
        self.chk_dens_all.configure(state="normal" if dens_on else "disabled")
        self.sp_dens_layer.configure(state="disabled" if (not dens_on or all_on) else "normal")

    def _toggle_substrate(self):
        state = "normal" if self.var_sub.get() else "disabled"
        for w in (self.e_eps, self.e_zs):
            w.configure(state=state)

    # ---------------------------------------------------------- ejecutar
    def _on_run(self):
        try:
            params = self._collect_params()
        except ValueError as e:
            messagebox.showerror("Parametro invalido", str(e))
            return

        self.btn_run.configure(state="disabled")
        self.btn_save.configure(state="disabled")
        self.progress.start(10)
        self.status_var.set("Calculando... (puede tardar segun n_kx x n_ky)")

        threading.Thread(target=self._run_worker, args=(params,), daemon=True).start()

    def _collect_params(self):
        z_layers, n0_list, gamma_list = [], [], []
        for row in self.layer_rows:
            z, n0, gamma = row.values()
            z_layers.append(z)
            n0_list.append(n0)
            gamma_list.append(gamma)
        if len(z_layers) == 0:
            raise ValueError("Anade al menos una capa de grafeno.")

        v_over_c = parse_float(self.e_v, "v/c")
        Q = parse_float(self.e_Q, "Q")
        z0_nm = parse_float(self.e_z0, "z0")
        y0_nm = 0.0  # la particula siempre viaja sobre el eje y=0

        use_sub = self.var_sub.get()
        eps_s = parse_float(self.e_eps, "eps_s") if use_sub else None
        z_s_nm = parse_float(self.e_zs, "z_s") if use_sub else None

        k_max = parse_float(self.e_kmax, "k_max")
        n_kx = parse_int(self.e_nkx, "n_kx")
        n_ky = parse_int(self.e_nky, "n_ky")
        if n_kx < 3 or n_ky < 3:
            raise ValueError("n_kx y n_ky deben ser >= 3.")

        do_zz = self.var_zz.get()
        do_zy = self.var_zy.get()
        do_dens = self.var_dens.get()
        if not do_zz and not do_zy and not do_dens:
            raise ValueError("Selecciona al menos una grafica para calcular.")

        dens_params = None
        if do_dens:
            dens_all = self.var_dens_all.get()
            if dens_all:
                dens_layers = list(range(len(z_layers)))
            else:
                idx = parse_int(self.sp_dens_layer, "capa indice")
                if idx < 0 or idx >= len(z_layers):
                    raise ValueError(f"El indice de capa ({idx}) debe estar entre 0 y "
                                      f"{len(z_layers) - 1}.")
                dens_layers = [idx]
            k_max_dens = parse_float(self.e_kmax_dens, "k_max (densidad)")
            n_kx_dens = parse_int(self.e_nkx_dens, "n_kx (densidad)")
            if n_kx_dens < 2:
                raise ValueError("n_kx (densidad) debe ser >= 2.")
            epsabs = parse_float(self.e_epsabs, "epsabs")
            epsrel = parse_float(self.e_epsrel, "epsrel")
            dens_params = dict(layers=dens_layers, kx_max=k_max_dens, ky_max=k_max_dens,
                                n_kx=n_kx_dens, epsabs=epsabs, epsrel=epsrel)

        # Ejes calculados automaticamente a partir de las capas y el driver
    
        z_top = max(z_layers)
        z_bot = min(z_layers)
        zeta = np.linspace(-20.0, 2.5, 140)
        z_grid = np.linspace(z_bot - 1.0, max(z_top, z0_nm) + 1.0, 90)
        y_grid = np.linspace(-10.0, 10.0, 90)
        z_fijo = z0_nm

        return dict(z_layers=z_layers, n0_list=n0_list, gamma_list=gamma_list,
                    v_over_c=v_over_c, Q=Q, z0_nm=z0_nm, y0_nm=y0_nm,
                    eps_s=eps_s, z_s_nm=z_s_nm,
                    kx_max=k_max, ky_max=k_max, n_kx=n_kx, n_ky=n_ky,
                    zeta=zeta,
                    do_zz=do_zz, do_zy=do_zy, do_dens=do_dens, dens_params=dens_params,
                    z_grid=z_grid, y_grid=y_grid, z_fijo=z_fijo)

    def _run_worker(self, p):
        try:
            g = MultilayerGraphene(
                p["z_layers"], n0_over_ng=p["n0_list"], gamma_au=p["gamma_list"],
                v_over_c=p["v_over_c"], Q=p["Q"], z0_nm=p["z0_nm"], y0_nm=p["y0_nm"],
                eps_s=p["eps_s"], z_s_nm=p["z_s_nm"],
            )

            results = {}
            if p["do_zz"]:
                Wx, Wz = g.wakefields_zeta_z(p["zeta"], p["z_grid"],
                                              kx_max=p["kx_max"], ky_max=p["ky_max"],
                                              n_kx=p["n_kx"], n_ky=p["n_ky"])
 
                Wy_zero = np.zeros_like(Wx)
                results["zz"] = (Wx, Wy_zero, Wz)
            if p["do_zy"]:
                Wx_zy, Wy_zy, Wz_zy = g.wakefields_zeta_y(
                    p["zeta"], p["y_grid"], p["z_fijo"],
                    kx_max=p["kx_max"], ky_max=p["ky_max"],
                    n_kx=p["n_kx"], n_ky=p["n_ky"])
                results["zy"] = (Wx_zy, Wy_zy, Wz_zy)

            if p["do_dens"]:
                dp = p["dens_params"]
                dens_list = []  # (layer_idx, z_nm, n1_over_n0, max_ky_err)
                for j in dp["layers"]:
                    n1_over_n0, info = perturbed_density_zeta_y_adaptive(
                        g, j, p["zeta"], p["y_grid"],
                        kx_max=dp["kx_max"], n_kx=dp["n_kx"], ky_max=dp["ky_max"],
                        epsabs=dp["epsabs"], epsrel=dp["epsrel"], verbose=True)
                    z_nm = g.z[j] / 18.897259886
                    dens_list.append((j, z_nm, n1_over_n0, info["ky_errors"].max()))
                results["dens"] = dens_list

            self.after(0, self._on_run_done, g, p, results, None)
        except Exception as exc:
            tb = traceback.format_exc()
            self.after(0, self._on_run_done, None, None, None, (exc, tb))

    def _on_run_done(self, g, p, results, error):
        self.progress.stop()
        self.btn_run.configure(state="normal")

        if error is not None:
            exc, tb = error
            self.status_var.set(f"Error: {exc}")
            messagebox.showerror("Error durante el calculo", f"{exc}\n\nDetalles en consola.")
            print(tb)
            return

        self.status_var.set("Calculo completado.")
        self.btn_save.configure(state="normal")
        self._draw_results(g, p, results)

    # ------------------------------------------------------------ dibujo
    def _draw_results(self, g, p, results):
        for w in self.right.winfo_children():
            w.destroy()

        n_dens_rows = len(results.get("dens", []))
        rows_zz = 3 if p["do_zz"] else 0   # W_x, W_y (=0 por simetria), W_z
        rows_zy = 3 if p["do_zy"] else 0   # W_x, W_y, W_z
        side_by_side = p["do_zz"] and p["do_zy"]

        if side_by_side:
            # zeta-z a la izquierda, zeta-y a la derecha, cada uno en su
            # propia columna; la densidad perturbada (si se pidio) va
            # debajo, ocupando el ancho completo.
            top_rows = max(rows_zz, rows_zy)
            total_rows = max(top_rows + n_dens_rows, 1)
            fig = plt.Figure(figsize=(13.5, 3.4 * total_rows))
            gs = fig.add_gridspec(total_rows, 2)
            axes_zz = [fig.add_subplot(gs[i, 0]) for i in range(rows_zz)]
            axes_zy = [fig.add_subplot(gs[i, 1]) for i in range(rows_zy)]
            axes_dens = [fig.add_subplot(gs[top_rows + i, :]) for i in range(n_dens_rows)]
        else:
            # solo uno de los dos planos (o ninguno) -> una sola columna,
            # apilado como antes.
            total_rows = max(rows_zz + rows_zy + n_dens_rows, 1)
            fig = plt.Figure(figsize=(7.5, 3.6 * total_rows))
            gs = fig.add_gridspec(total_rows, 1)
            all_axes = [fig.add_subplot(gs[i, 0]) for i in range(total_rows)]
            idx = 0
            axes_zz = all_axes[idx:idx + rows_zz]; idx += rows_zz
            axes_zy = all_axes[idx:idx + rows_zy]; idx += rows_zy
            axes_dens = all_axes[idx:idx + n_dens_rows]; idx += n_dens_rows

        zeta = p["zeta"]

        if p["do_zz"]:
            Wx, Wy, Wz = results["zz"]
            vmax = max(np.abs(Wx).max(), np.abs(Wz).max())
            norm = CenteredNorm(vcenter=0, halfrange=vmax if vmax > 0 else 1.0)
            for ax, W, name in zip(axes_zz, (Wx, Wy, Wz), ("W_x", "W_y", "W_z")):
                im = ax.pcolormesh(zeta, p["z_grid"], W, cmap="jet", norm=norm, shading="auto")
                for zl in g.z / 18.897259886:
                    ax.axhline(zl, color="k", lw=1.2)
                ax.contour(zeta, p["z_grid"], W, levels=10, colors="k", linewidths=0.1, alpha=0.4)
                ax.plot(0, 0, "ko")
                ax.set_ylabel("z (nm)")
                extra = " (=0 por simetria y=0)" if name == "W_y" else ""
                ax.set_title(f"${name}$ (GV/m) -- plano $\\zeta z$ (y=0){extra}, N={g.N} capas")
                fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
            axes_zz[-1].set_xlabel(r"$\zeta$ (nm)")

        if p["do_zy"]:
            Wx_zy, Wy_zy, Wz_zy = results["zy"]
            vmax_zy = max(np.abs(Wx_zy).max(), np.abs(Wy_zy).max(), np.abs(Wz_zy).max())
            norm_zy = CenteredNorm(vcenter=0, halfrange=vmax_zy if vmax_zy > 0 else 1.0)
            for ax, W, name in zip(axes_zy, (Wx_zy, Wy_zy, Wz_zy), ("W_x", "W_y", "W_z")):
                im = ax.pcolormesh(zeta, p["y_grid"], W, cmap="jet", norm=norm_zy, shading="auto")
                ax.contour(zeta, p["y_grid"], W, levels=10, colors="k", linewidths=0.1, alpha=0.4)
                ax.plot(0, 0, "ko")
                ax.set_ylabel("y (nm)")
                ax.set_title(f"${name}$ (GV/m) -- plano $\\zeta y$ (z={p['z_fijo']:.2f} nm)")
                fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
            axes_zy[-1].set_xlabel(r"$\zeta$ (nm)")

        if n_dens_rows:
            for ax, (layer_idx, z_nm, n1_over_n0, max_err) in zip(axes_dens, results["dens"]):
                vmax = np.abs(n1_over_n0).max()
                if vmax == 0:
                    vmax = 1.0
                norm = CenteredNorm(vcenter=0, halfrange=vmax)
                im = ax.pcolormesh(zeta, p["y_grid"], n1_over_n0, cmap="jet",
                                    norm=norm, shading="gouraud")
                ax.plot(0, 0, "ko")
                ax.set_ylabel("y (nm)")
                ax.set_title(rf"$n_1/n_0$ en capa {layer_idx} ($z={z_nm:.2f}$ nm)")
                fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label=r"$n_1/n_0$")
            axes_dens[-1].set_xlabel(r"$\zeta$ (nm)")

        fig.tight_layout()

        canvas = FigureCanvasTkAgg(fig, master=self.right)
        canvas.draw()
        toolbar = NavigationToolbar2Tk(canvas, self.right)
        toolbar.update()
        canvas.get_tk_widget().pack(fill="both", expand=True)

        self.last_figure = fig

    def _on_save(self):
        if self.last_figure is None:
            return
        path = filedialog.asksaveasfilename(defaultextension=".png",
                                             filetypes=[("PNG", "*.png")],
                                             initialfile="wakefields.png")
        if path:
            self.last_figure.savefig(path, dpi=300, bbox_inches="tight")
            self.status_var.set(f"Figura guardada en {path}")


if __name__ == "__main__":
    _make_dpi_aware()
    app = WakefieldApp()
    app.mainloop()
